# E0: reproduce FLOCK zero-shot results

- **Kaggle:** select the **GPU T4** accelerator, turn **Internet on**, choose **Save Version**, then
  **Save & Run All**. Download the `e0_*` files from the Output tab.
- **Colab:** choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Keep the tab
  open. The last cell downloads `e0_results.zip`.

The notebook checks the walker, runs three small inductive graphs for comparison with Table 9, and
times 20 Hetionet triples at the paper's zero-shot setting (n = 512, P = 2).

In [ ]:
import base64, io, json, os, subprocess, tarfile
from pathlib import Path

EXPERIMENT_COMMIT = "2b10143869581efe780d4bd3fbcec440a96826b7"
UPSTREAM_COMMIT = "f35103d25a78bdf4075de5c673a51de4979aa4d7"
WORK = Path(os.environ.get("E0_WORK", "/tmp/e0"))
ON_COLAB = Path("/content").exists() and not Path("/kaggle").exists()
OUTPUT = Path(os.environ.get("E0_OUTPUT", "/content/e0_output" if ON_COLAB else "/kaggle/working"))
REPO = WORK / "flock"
PYTHON = Path(os.environ.get("E0_PYTHON", WORK / "env/bin/python"))
SCRIPTS = REPO / "experiments/e0_reproduction"
RUN_ENV = {
    **os.environ,
    "EXPERIMENT_COMMIT": EXPERIMENT_COMMIT,
    "CUDA_VISIBLE_DEVICES": "0",
    "TQDM_DISABLE": "1",
}
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)


def sh(command, check=True, cwd=None):
    print("$", command, flush=True)
    bash_command = f"set -o pipefail; {command}"
    code = subprocess.run(bash_command, shell=True, executable="/bin/bash", cwd=cwd, env=RUN_ENV).returncode
    if code and check:
        raise RuntimeError(f"command failed with exit code {code}")
    if code:
        print(f"FAILED with exit code {code}; continuing", flush=True)
    return code == 0

In [ ]:
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
).stdout
print(gpu)
assert "T4" in gpu, "Select the GPU T4 accelerator in the notebook settings."
sh("nproc")

In [ ]:
if not REPO.exists():
    sh(f"git clone -q https://github.com/jw9730/flock {REPO}")
sh(f"git -C {REPO} checkout -q {UPSTREAM_COMMIT}")
BUNDLE = "H4sIAAAAAAAAA+19+3PbRrZmfuZfgeKt3aVmKAbvh2u4tR5bTrxjOy7Hmbm3dLUIREIiIpLgEKBtzdzkb7/ndDeArwFSAhU7M1sjlssigO4+/Th9Xt9pcJN8iq+X+WWyjBdpMk+3X33+j0kf3/fFX/q0/9JD9yvLc0zLDGxHlLNcz/K+uv4Cfel8dkWZbInkNs/Lu8rd97w9uP9PPp5tzPLVKl2XU/vSool3Qj/yQiu9SoPQnLuXc+fqcpbOXNdMIj+0/ctg8I/u8+Pn833ST5t0m/H6F19/KRq8H4LAO7z/6Xtr/zuu7X7lfakO4edffP/j+qdmvE0323y+m5VZvv5s/PCA9fc923tc/9/gc9f6zxbp7Cb+mCxv0u1kc/tgGlL/uwfW3/Jo1Vvrb9Ol9dVvMon/4us/HA6f5atNsk2NcpEa23SZJkU6N3jV6WqWb+fZ+tr4mJUL4+lmk67n2Sfj6cQeG8l6bggOoYpJaRRpOlf1Cqq4SZNyMhi8260L1XCyNH58scxnNxNR5kcjX4sn19tkszDyK3GxSYgd/1dhvMiud9Qli8iU4sHf0m1+Wizy0vg2ZeZMmWJZUt8mg/f0fJ4UCyK/zNZEPlvLPl3JVnhwf90Rm6fF2Chybu9W3FxTczT0sqIuujIxnhrP3v5gXO6y5Vw9GaTrD9k2X/M+ka3TiIghjHdnT5+/PjOywkjX+e56QUP+oUiu0yfG5rZc0Ahbe8g4Pz3lmTIs03Yv+CrflZtdaUwmX2/TYreU21CWj0XlyU9Fvr4Y0EINBtlqk1N3k+01dbtIq+sFDX6ZXVaXXKH6nhfVt80yKa/y7aq6LnaXtNdnaVGXKG6LwdU2X9GclNyeoe6/pcua9k/Z+qfErq7WuxUNKimM9aa6Vebb2UK7iK/TfJWW22xW3b5NVktJqlVgMk/KpKL7nL4TC529/c6Yik6M4vgqW6ZxfDKhycqXH9LRyYR5l6TXuX0xePbdmxcvv6HCw2I7i+luVt5+PcvXxAdfMwMx/8TlNlkXQsR9SL9ee5ZNBYt0dblM7Qn3a0gUXz19//K7N99TS+fDZXaTDsfGcJ4V1dcrYqX1/C+0J4YXgzffPT+TJV/t5ONvk/X3+TLnr2cr4ud8y1//I58n/Pc5Mdziz+xq8dWzRfrxMpnNxKP/m1xeii+v0iyhps+efyObHhj0GdUEqn5UhE7GrQLQ16oHnTIwiKp3d7UD3e4Uq8o0g6mLwFxAY3KgdSGYpcOF9HmrComZqstos9kucjEYDObplZIKsRWL7T46eSIq0/76o9jxjUxKpOhiliRZY8yWSVGkhTHPnxhpMlsYV8mMt9WuMBLjapmRdCSJmNN+ENKyvN2QkDB+b/zXu/+a8O5lKuxjF7yigmsm2XqefhrxzROD9qZ4PDZi+sdSRiz/hahXJtmyU49vynqxqMTX7XrUCa4nttmkJE7Pt6PzmsFVQyT1ExaqdWPVDa0jJ6LFdH6dxqJap1kxONmNQpXepuVuuxY7eSRu6E1MZQOzpBydN3fHUGLCEzsyTy7GxjxbTS211HUzPEBsRQx4rMb9e2OZrkf1aE8uoDZJrnidz9NiymXEtLaeVpNQTLVh2sbvWs2qeieKv5Z5Mo9X1PSy4i0SrBOWqTTdRbotRyapoXI7EpLtaxRWwxM5bUI0XgllKRoqKpEoFOhALm26YpGe0jJImTx5r+6MqpalPGRhST0q00/lSLUvZSLVZIE3KZKrNOZej6o2qQZN/nakWH86lPrYou10vdkV03OT1iPd5LNFMaXBXNIa0J/ZzYaKrkk5V+NQyy96PZJEz4diRMOLk0n6IaEpUrNWJEQ6HYmHY7Hnxmo7sL6cvqFW5ddY6Jzpi2RZpGp+syvxhNUwq3Qu+6ReSrl4q2S9S5YxFxvxf7J/682EtME8X03qB1VjkoxBc5+KBmUFubNFi+ucBci8WmMYrhiDtHJGciDdfXJxUg18nl2nRTmStpZqjBV6uqX1UZp9UiwS2/NHijtok8q2eHPKik0nZN3JbjNnVpDlJmIwpCzL/PK2TIuRvj6qyiL9pDpT9W2Rb2Kyj8usKLNZofo4FsbUbb0/plLSNnL0NZmQbHct8o/cQiFsLVk3nU+Mt3mRcT3D4gUrbqTgvExnyY7mmnf1KdvkZW1NXpHJmZW1DBU3x0rmxWqvj411sqJVUxfzjOiJzctCU03CFfWXvvL2s0+qKdk7m1J4rhsRQK3oe15Nn3xMT4mTrnbL5WgE0sRovtPFqSUrJdcJiYFS1hFWyR2V5kK+Xeb5sll6uSt4uC2RDzzAGz6by+f0hYh1FM7Y6OgSYGQ5sPN2QxfUUv1Q3RtX5PhhW7VwN5t21dj31ny/3aVSsm226Ycs39EyznZbNu7oqVj08ydjw3pyapHwqa/tJ3KxmFVhPepOdhprijcrUfXrQOGKd1njMX8h3awgRfUhJWtcUFUF//cUuacqSKbJzcdky+00DKoaM6ZTw6qk2UgbzR8M82SSrJlZaflZwo261f/APJ0sl5o8SjLqFvlgJXn4Z1sysUbDq3xHjlvCEyCdJCU0E6MgZ2opN2DlEFVumDSIhidyfZJlvmbl8QtMotRztNi4CDgK7icvHY+zfd9ScynqC26FzSwHqGq27juqohRIQtZMW5zQZUldemmC8O/11A25seET2lulthiTIvtbCobCsFryukRMC01zx36GYIt5zArxicGbeDSqOeR/6OJF77ZaSaQjdkqZx2LrqM413W11JSY5wYxGZciKILto9EvDp+di/S6M/2n8Ajyp7p5MVmmy1ki321bN1DUbIvto9CLxMwxUDJAGKgZ870CrznQGDH35Zf+Af+k94u5If9nbfL/WcbCk18kVWSXlbEG7L05UhCVO7IbWCEXMFCXXCdEcaXJFf7yH+LAkMRvX4iPmUVLTCbm/MVudYm/Fn/JtU6bmXezI/8PBnrQJd9mXjL8ine3Y6Y41MRHDRtMeTIrdqm8bchrZOJMPEjJCb1ckyEma1BOppMu5VvVizxTpAkJ0MCbb4fo6BQ6o2OO8kT0XPaRNa0l+rqxftjhFuKc4ZAIruV5H6MiuUdbaPaazqKxcO2HiJOvrdGSfKJkr43ZHtaYZ4qy3DzffFawy4iMayNdLnhqOFTILVEMj74Jnsr4kfd+uTka7JC8batqQw1EtqAutvrwnImxFLM1qrKYvyyotE56BESwA37/OytHvku31TgTOux5AE1ubbHfr0fmQKpDv1FQhC2b2cT5lP42+JZuSIxIyFCgmlAwuctfUV8EXcp4nRTmnYvRnS26x0sf/ZnAE9E/EoEsR00xpu94Yu/WGtieHMEjDz2g1Dd7opOdnNK1rY7ehNtJkRZekn0Q0dyPjG1lJVnm+285SAQ2T5S19Ru4FzRw7ssKqRr91bAyFiXAqI5fDC+VmcnUqmxcTFUGdXKflaHj272/P3r18ffbmffzsu9evX74fnlSmj6pDfdbduLotnnsSzh9ORQhUhMHOnj4fNkam1tGJFKmj4R1Ax/CQDSBpEnvIL5oomqfs42dXGVEiy2mWxnVpIS1FN9l32hXcx9NT8t9nJAWytbxkdtB6qokg5jfFlroAJA+a7kvHHp9sdmR0sJ9fxMkH0qAs0KkgOxM0+QVRmsc09cnVVbamBRuZGjkZrKbyVZB4Iu/ELOxphkYn7R1IZaVTG1dl4rizS+uwblUaAr3764k9TaXJNeoUqDclzWFl5QoeYEe5ColPnqot9lY8Gc3TYkZbRfipcTzPZ9Qay6oVu4LbWMT0pnXld8nH502Fb9Pl5kVV9AToTZL5PK728mgoQ/pD6YlOSZGNWUgku2U55TD/nTXlnlccSA94Z6mi4g8XLqqNLmQyu6IQYZIyiaPm005ksw45tvw/GbituJ71GBW4R+5zPyb7hb9qR6AX1BBsoEp+0orWorRuCHmKxUYsvXAWyhvBu1kdFiHBPBG3u5Y3mU1Udm+gor2hZNF7VG27ez8rl5TtEwZWJnMaOFMQwx0bPKOcPHNSSzDRglzXRnwJ+AKenEw+brMylVG5O9s1fm8M/5Nl1GBAjccxu0BxzOptGMe8F+J4KOnIjXFces5d+G91kcYcpogZPXkQDHwP/uv6rt/Gfx0r8B/x39/gMxwO3+3WOvYrdbrUo5ucGB/QV4H6KsCY1LGIh75nZWNEGoYrgNBUAii1qaHCyWMlyLitkkxOaZ3s1rMFm460B2WUFbukAtZsZAroRSK1gx9PT1fJp1M2hpZp8aNR0LZPGY6RUV22wwS9j4uMKpFNsSvSq91SxvyyFSPbl7tSxD/koMRIcFA1oCvFgAR1928M4/RUDbAw/vLGCt+9e7lWOOOTDxY/TT9kbFRtdnRxyT7fKUcTDMv4z1pOHACEGzoC3H0gGLzMr69pzNWlnKT6KpVGH6LBFYpLAuJ+ZPggFixqIhBeVa90wpeAgr89e/ant9+9fPNeBCUl0NJwdPG1QFeU+TrZlIvh4OWb5z88e//yz2dxLxw5W9cgsuXXGLLlKxD5/bunb74/qsH7gen3T//46iyO4jdPXwtcWGra4Ys/Wt6N7QTIbqTo6P4pfZFqbNjlSC7ylzdQ5M3Zq1ftAm9eQYEq9YIf1N/Hg58H7797dUbjfXZGnTInpqUstWrjfwYcrGIE8v6XfSCyugj5X9UTljUDNPW5sbGsMFZPG288JfOTtjvtF/T+2nDRQRxJPrgLfFKo5W6eYIGYQyeyEMCJkldGssMy9M/bI2atxNYmC5axIWRKLCKUT2q7UAhAMnqVNa2Cq7xXSgHDjIZPhpWJSnJvauzjXLI8KhicDY968SU+1t45FcbIxo1uEKpGiIGqrjXPGFlkziJjbygISnHJ9HiSFC1yEZbLIVSTMCRVNOHmpbAgO2UZoWQTkJiuERCaoUd01USh6ylGcj5UT4YMV6jvcqhXHA7nxZngaim+5mkdV41M1d+TquJETcRE5BNNRd/qpW1KkXDI1hPm0bhZZSreXCBjU42xoFunPCzJheEIo/QMxPOaOUSrfHU4FYJU+806/7gWOQ+FkX6iP0sh7H+kDUZm6JqE6O2Pxjwn9Su06yIt6tSJosbu2MHn0STFzfmQ+XAoAjXDWuy8XF+lJMRn6bATVNGTCPizP5GgHs4EUwqau2TvkSccNw/35BbUrXfyC/TG+fGhtvnZRavNJuugqVPfa4r2S56oOOfAsPZlShyuIkYy6PZT1WAPrNXRGrknI0v6jvWYyMJLPsXKLFNcta6uGUU7vBjSzzu3Lip+gZaqqBDDYHhboG3qosM2NS3ZoJa5wkKbjNXVqK5+cv4EWlYhUmFFat0WQTPlYMvH3ZHcPU6GryT3HWpEJBAdaoMfnkP9JvBIzaiFEYZ4/CFZ7tJCArFycko2bXkZhN/JEquoU0a0OJlsIFkny9siqy5lbWGHDveklmzzj6xi+A5/ZVnAfzlpUlQ8H0a052ld6e75cF5IAaBZNufcVz2B6O/D1XbLIWKu9GJIO3a4yMrCMut7i+FFFSDivJIdJz9IRclet3A5dg0zSkUqbUAVBBCeyLQqJ0qxvSyiS8pw5uDlK3FvNExNpTXbxkITQKiE91i5L7FSsIc0uqgJal1eK90uLkDBq1i0VJFS9zBzyDss4JvIkGReUUQktcbqplIDosZU/lGWScVyDFSr/W9fTMp8hKWEQoilQpj2VjHtVqrFEiGofZJEDBzFiR4LgzWU4ZyYKE9ZyYirk6aw1M0cJWIAbFaOpAwQiU+NNcCiaxNToyq3hby1ITmQaXa9KAuBWKjUoyaRSeuPJNUaZBUVaps07SwlYQySP0MLuUkTGla6yre3os91EFDkxrBcyNiKS7dXxEo7sim2oz4pStK3qmSwXCW1AeqVaAxKuQHGanFTyTlTWPix2qCxahcCgkVKLC6SHff01DiV4xh0VEPdi/tUA6fbxGzdT7H+EbplJdOUmPEq8SKxPTWWc3GTNDjImlYBvv9/6MHFiQoTsoQUEdy25D2AMDT2sBABe0L9IDpa8dIi5oh9zeki4YzDsmgbKwcy3mglxRTJnVag0VxLFyrdkjcY2+faasJpXuNmDH2m/lBD9brzXFR3oXC13LwI6isCGFJT8f0qp4LRDL4LpaoVF+Fo+RWfbhn0vqrtz7qMYgTiWdGgusTVyFj0x2W+TMmiEHWTy2J0d/0T4w+k92rHmUREzdF10qEWwBbbSQSwxbfus1ioZzFzTTHS6fumU4gXhVFWYub6UoCwMgzE2mS749DXqL717ofvn35zFn9/9uoFaf4dmf2ftgVTYJjjd7+z2+2TR6e1rXm8n6pHtGosbck3Pqmbcg6KzPbk/BOjQpX3M2zwoL2xmnE3NjPuxGLupiQmachQcU5fium50ltyzi6gA+LGnY01kcn9kNadlSEcC7W/EHjGE3zKtlJV/a1wtav6+0xarvL1zfXdM6BAOWi0Nz63k1qsiXwpr+MqZ9ugi4HVjyebfDOqxH5z+xyCllW6wkU3MbixX6RFXmX5ajm9TZuKXy5qR0jsycZ+FCDTyDy5ZxcKJmujfwj6MTViQpJpLLjOL6SWZJ+ACbBTUNu8IkxQbJYMm4+HWkqFaPpctnJRAfp3GvlNMsAh4K5q7lxkk9ZgG/sEu2IBZoyYtAOgXsUT6sE/CaR31OcOd++z0XjI+V/X8R/P//4Gn2Pc/YfSuBv/paV2O+d/bdN0H/Hf3+Dz98EwYtE8oC/CumTUyPaG48FQCGO+U9shBgnaLT/6gcPtEycM+ftCXPgufX8v70cOf1f3Pfr+Qnx1TS7zQt333cHPY42sZ/Yj6wBZz3Uauo7b0PXcoCbseHZD2PP9NuGg33hd04Hxms2AXdOCAVtWM2IrhBHbdpuwZfYbsutGONN2Q9nxgbIDc41D9n1Ho/yXP/VeYsuHuXYawo7pNYTdyK8J2zBiN7LadHuuseUDXcduplo+UHQdy6npWmImFGE7CNuEe66xI/i1HnDQDNgPgbmAsBMiczkdwn3X2BI7SFG2Q78ZcgiU7ShqhhwGDWXH1PfTm1enPbeTYKdqxDZspxCW2HMjGLEFvGX6bbp9eSvCqfajhnAQAOEQCLseTHXUptuTt1wzALrNCkvxVI83bMj6wFqe77Xp9mYtTW4Ba3kgPjy72cSOBRPt2Z0F7i0+AgvEh6QmH4QoPoJmF+viI9IF1+v3llEmn3qRtm3gaiU01KCBqxU5xfpA2gmdDulFmiz50Hof6hFMuROAigp8pA6qIgQWc0OrRd028u11L9ImcrflNdxtRrDaltfMOay15beHbRvFLOs3Zg/H7MGYbVDLro9shjLbC1ukHfJxyp6kQU25ZqMt7BC2lus2CtIxYdRuSzMzafIgt0kv4r4FxAOv4TTfAzkWRM1a+yHYI6HdXmu3/5QHwORSV6txRzjjh0bdXmv3GBb3cYMFoLB8lKLkfsGwG+ItPfmaM8STVb9BO5r1B3zmAoOHFtgkAcjRyPFa1hCHvPqJsxC2lu+BNRSg4dnMt2eCSAnaRpjIb9pLFUiiIPG9Rjl7FjCXHzZiDMSIH3Vp2vfT9CxQzM26SgVRrWtjh3gOkCRG6JB0+gxTUxTNmnqahes3KtlDFeWH7TU9/eD2oBqCsafYVnUH1zMMYEFxrKaunqqctnuo+i4saSCtO7VcsGuDqBFWfgQ7JzSdDtUei0qmOFIFKRWAPgxCpAq7Jog6RHssq9R2lXwKGu6V/klt8oDj4oNp6Zthh2qPZZWOUDVUUP1y5quh2g0L+54DQ7U7Bl6fZQ1QOERWY86GJqj8MAqBKi5rSxoz1T571fZxrLBbfWQmv6HqhSiGWxKCqfZYVyl8aqqNIPTQuApcEBIRyv7uUPvs1gD1rAVEHSQK9oXn4W4NdSX/7WvDurmfqommhRk1VM3AbqhadjPBpg9jtayoTdXpQ9UBV8EEyWTislpWBFQDpBq2qXq9qLpI1QKqOFYLiLpItDPBRCW7znssLdBVm1Te16IMLmh0Gzas3/J9X756+8z4fpWIFMz7JJQJus7FmI6JRqsHRqsJy+u2nEFB+hVj0PeTttFmhUiDY4L95AJj6eZya9BsxESR7hC+h8RudEMxtAJDdpGjfbvhLTfC+JnndQjbLZ7eT9h2NCu9WWbNWgU72XZglZ2opQye83sj0m0PwjJSUhGOGsLSiKhGbMKINf9XJ/wsX8/STfkmLckN7jNuE8lbfiNCLDQzHGBv29UcI6dF//nZpz38fWC1AyDu+8GB1W4m3cWxk3m9h3aXwQ/MO1rqLgZaHAzi+RhoAYPZbUVq/+PpN985ltln0B6ak1ZjAbgmsJpvw6gtZPFAJ/x0mxVl/me3B2UL11pGFKpgGrCajLJV8gTX2tXX+vkfN+k8S3oymoPOgofmlo1y1GzUMsa1aDp0+d0ctLh3Z3ugrpwA/NAAtzbEWlywLZ3AalnvlndzajvByDJP+lBHg9qB7S2pVPsrcvaTd812eLwib/cjjztMaSz5AHU12EJ24OCKt+GIirrXi7qDbri0Bquoixbcg9CHBQrMY0thcEE3RJrZA+EYhCccCLUdxGOQ7bqwSG88BofuNOt+Bx4DRpLX8VT7wjFAFbjtMBhj43CdNtX+WIxmJYHe1rEY2N0+TnMHIOi7vLaF0wzLi1ob2Bu9Va8VpT8CidFQiUaOHsRhbFhbu0O1b6wcJShYg4dAGI2VPa9Dtj8IE+JwjwdhOmBIzz2EVqgHEfqDIIwOLVodX7knV+lgSARy6xAIo0sNuzPgviiMhUscoW2AI0aLP0Jr0LfbhPuyFgZrvT4ojAs2iec+HIXBwLyMPVcoDJqAEHCSgbHKcbZ0ysegMA5oBhewCCfCRTYbH8e1UDe2ohNHgjC69QkTfhCDQT+HRNqDMRjLhKW2wHuPYDNLT0BtZhO996A94f1BGAdjXy7uZ0fjMrA/NZDR9lqk+4MwjqkF+xrhaYdoByEIg2vdAdyOwmDQAA1NBCOAxSG4GaAXHzpt6Kk/BONgboYbgBGiJSpAgF5abLV+jDqkj2DxABlNRiDfqwcwbvlADRzj5q2o7jEoTBTVM25NzJpu1GxsefuFul37dXT74fgLunU+wMkYw/ZBdHsOjtbqZP70Cex6JubAQIT1IALjYrDTbjs3PeO6uLDNSA+CMFoE2+sOtE9UF5lJ4gX3gTCYdONHHUO+V1g3QhAGtdNBEAbX9IEYDAbrg6gZaoC7NoBgfWCB9RFaVodqHwxGi2CDm+yH2m4FKYX4YeR1iPaB1lAjeC4GBTQLD2xaLdzXigr0xmCQf5swiI/Tazd7xvcxVO9abVunFwKjKQEQxSEm2IRgQwc+zG/UivP1RWACZCUQSl4AuyaQkXI1N0A17FruvfaqBudB0oFO1WvG6ps+Un0YBOMjAzsgIdCGDQKQhYE2Vn1d+0IwmCgGoMRBACbAjJZW9LQ3ANOQNCGHph/80qXZC36xgZNMAIbNwEWqCDXhSDtEe8MvDsaInWa4h/EXzQl0dMrH4S+ah9ALf8FEqZamOwZ/kYHXOm4HpjLwsrxfGW5AuC2iesMvujkBCR2mNtcA2KIN47ezAPujLxhhcCGKg3Q950BCbYvsEeCLa6Gjb+OANbip4S3MJ/HcNvhxJPjioPazwQPFECEIZz1lu438HIW84DpDRMfFnDQt99FHUNPdR7s38uIe8kowBcIFh6iFvOhKvz/y4qEORrsVc7OUWaMqHEyROgJ20ceL0SRXSxfHXGKYbM/+NbCLlJJ1ej4EZj2IcvigLVxMcmnRPgJ2cSIM3mEAPELudoC7EeZqb+qjcRcPsw+tZrFtD2U3RHdshF1+LeqipT428szGXDyMNtiYddkG7o+HXTCHHOLTjg1SzQML17EQe7Da4IMir5F+u03F65f4mGpNWEvmxsMvGBj3PMRUkdOdtr/EJ4DvpYoImzTtKx8YZRlgHph547fC4lKUvaYdtlvdP15E15T3q8aL7gviW+gfuuyUSnyLXyv3IHxL+p71lDddCDVn0W62WRihs+i0Wa1vqNpH11j6GcpLxXAHCDVM7Qo7h356xql9DUUG1yJE1g4AVotM4LCwA2j2DlQjjhjAnooQygwbcRJp7lv3BEzP9Q01ndXojhAh+xDSBSNPcxujNt2eyxv6B4IQIaJNIdj7ciKqiW6tcH+QK8KQQAh8FVmYqhg6B2a6tcJHwFwhbqUQcvgiTHoKIcQUoVEUOZ10xZ47CT0cz8R4hAbsQWgANaUXdMCX3rJDE9cN4cBF69dqZEdgQ/TQdzpoU1/ZodFt7N4ADyX4YO6HiLwE3VhIT94K8PiJD/mSQXhIaDla6LITDunLWwHagKgZQzS4AzgyGDkIAoTtSHx/mCvScn9BhEQBzHbkwU5GkDz6dThXoAH0EIGK0BSKAMONENGMOshLf6ArQissArgpwgNeEQSqW6QfjnRFmFse4ah9F0dtAWkP57x9tqw/0hUiDhFACkiIoHkIUG6EUC7Jul8BdUlRXI0b0jHQ5o9MoB1GOOMPRroiBHPDCCcc19ptRJnkjoqy1x71UUhXqCW6W41AixzUHBFusTt4/Aiky0IopkG6bDxOZ2K6D9C17Xa6T2+wS7M5G2EW6Kl8zTLrYnTPMZxeR1PQ4LQPwCIeCm9Uk0FntP1gEe08bgOwBZaDRBua6D4Hfhdf6xE/16AYD+KOul4GZtZSIsIOzT6YiJaQEDZiI8AN5EOiXIiZgkHUPYPTY01DjE7ASdQg0gwQkNCIT4de9whOjyUNPTTlwfpA2ewBTqATbb82oB/SFejnq4GRMPjkwVkYKTtqA+BBp43wDFkIwZ/WUA/R7KI/ffYprqk60CwfYMRJhtMr7sUwm9PF13osaoA4DB5/1N1RMCkj9Avb5wb6AV0+Qk6ei4IQDVmrYaUQkdp9Q+2zVbVclhAYGLUspkyFeLo1bGn4YxARqborhoI0z8gC7CnEvYPBlqiVb3AMIhJqUGbUjDrEUYdwMiTCgwqR2Tl11Afe81Gzu/B6ApWCoB6Ai+Rj6qPXptkH3vMiLWcbkHjctS643ro/6HZOdfUB+HxcWhc2kJalhEd+Akzh9aIHI3w+WqweJOKFHsZ0QPVEHp6Zth+MdmFMx4VT4h4KDUzC0w7Hew8+bNSSG6BsPVS2QDjUcoVaaZ5HAF6+/rqNxhnUj1eB6LA1y+1XnjaycKnNsJEfGC/FMzdBpMVq98BOfREvmHE4+aJ8M3RYKscEg0pBl25ftCvE3RzCUboIIcbQASWMhlUrr6Y/2KWZyy7q/lCzXMHgwNSEqIVtHoF3adCLbWLaNigJG8/CRiAz3Vaq1nF4l/amIkiOCCwtcxsUFJ44CVsG+xF4l68d+IB1DvFYlzQDK/0EhA8hPv3QLj2SB7H4CEEA8AMjzJEIg3Zy3HFwl+4jgcUXIpAdYIRJD1K3PbPj4K5AA/HB9w8xlzoEKyxCeyhyaxSGf1/4gSgMLj5kJx5EYSItx7d73OfLoTAYpX8oDBPgFvcB3NVgGKQbYWpv2M0B7R2kx9OTAcSOD+MwaIA6HTykLw4TITjgoBjHsBrEbzUe8zqoxK/GYTQACBgLD3S1d/bngGEwpAU+hj7P7oPf+BZhRCsE97yF/4CqDsb6Zv7MIAxGl9Do1lKfzA4W0pOxAnybgm+D46odBzYbzgrwoCCpnQeCMCGmmAXw4sYAM/p8ANpa8bSO8/ogFAbyf/A8rh+ADYqOXNCFu3rLDu1VlcGB0HgAxwVb8ZAuEtIbhAHzM4I88ggVNIpLLUDcCYwfBcFEeIwuhEMCd0AwGBsP29SPgGDQ3o9AL0YhSpEg7E+6PwQTIGlABIDLIhScnwuBiTDUFwLuFOKgQxvNIaBs/Zr3vUX4itDIxbWGUcOLBSN8kVLUOVZ2BAKDJkEEyYwRupYaacxyi/wu+HMMj6NZECECo2N90aHlbr3n7ggIRkt6siBtwEan1kTbT9NY0YNf+aZlHwGwq4fsHTD+AjQ6zS440Se8i7a+1QeF0V501D1Q3+9wClJ1e8AwGIUj0/9hMAz6FfB6u8MwjDa/3entE9zFyNChJUUURntdo9k9D9MHhcHD+4GHlgCm7aGxh8mS4Z73r/WBYdBhDnwcK3JSgKYtLqr/oLe+HcZhPG2sQFUTzt2zVb1gGA2TxyQ9NLU8mGDMiwyj7uGfPkeO8N2fHhh4HmhAB3IQQhTI7Z3aE4dpBRghmIuvM8JF1RzDB73xTQ91gXUVaDnOLiJO2nvmuuBPj0XFxDwXxL6eiYghRe2Qk7vn5Vx9URgbQ4qY0XMYhUFVa+15JdmXQ2F00h1E5MugMAEiBF47qvfFcBjMjmunzn85GAYxiagz1L4wTICTjHnUIb4QK4A39kYIAEXtU3u9YRg8LYgWxUEUJsBsQK+VpnUECqMFTeE8iJ6GCGHL0NfspwcfOwq0n0wAGwpjWPgGsBBfwB22NtHRKIwEpPflelh4PsCzDtHf88633jAM4vRgput5ceCBY7wy8n8FENNyyxrSJiY0uxhX+hw4THQgMS3UcqXgvcx6OEu3XD8HCuPiez4cfKcunk0gI/tLoDAYlUZAF1HGtmV3BAijB9MA3oxQKYdwoC/Co2Vh553UXxqG0TIvD514+rUoDL5pLsBYC77jN3IOveiuJwpja8dw4IAEHpYNEQTSGL2dtdb3zJEWTQxMBATQIkF4FbWW1/aYeh05iiwtSA2qA8FVHw636XiyvwfI7nnmKEKrPoQAj6PhHyhPtERXm8Guwc//6J9Y+qf+4O9/bdO/7rJtKi5Orze7Sfnpzp+86vu5+/e/TMvyvPbvvwWe6Tz+/tdv8Pk345u3Pxjp+kO2zde88uLXGcsF2csNZ4zFj0rnu9J4e/uefy9yLH828rSYiV9BNZL1XNThnyDm34v+uMhmC2OdpnPj2Q/Pn54Wm3SWXWUzQ/z+eDEZzNJtSTemU9u0/YllkdoezBb8g5rl6Zp/XHWZ/S3dTqfOxJs4g9ky38032exmmfIti26lSXHLP+I9nVoTyx1cZcuUfwh8OnUn9oSuCyapmo8m5iCbrxOuS3R+ytY/JbZsyB/8lF8us0tuxqdiq2R7s9sUyZUgREbDYLVZJeWCn9P1YJ1sPy6SZUEtT2xul5T1x3x784mLB4P1brW55bJE1R1sktlNwr8bT6V9ampT8G+wT6cBddEebG4vs/XcsmRHTLrmHxQVpR2iRcVvb5MVFad+UWO8OdOiFJQdGuSgmGU3WXm6TJPtmklGE4vvSfpWwFdpuduUec7dDUJBpLhVHbRcuioX/GOhGyoxK5dMNuSbYmWv01z+yjXTE2X/Ol/x7AYmtVzebqijp+mnMl0X/GPL/MTiCdxtlzSdDtfixj4u0pRaZtubrv7RzP74efw8fh4/j5/Hz+Pn8fP4efw8fh4/j5/Hz+Pn8fP4efz8i33+G/yn9ZgAyAAA"
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as archive:
    archive.extractall(REPO)
print("code ready at", REPO)

In [ ]:
sh("pip install -q uv")
sh(f"uv venv -q --python 3.12 {WORK / 'env'}")
sh(f"uv pip install -q --python {PYTHON} --index-url https://download.pytorch.org/whl/cu126 torch==2.8.0")
sh(f"uv pip install -q --python {PYTHON} -r {REPO}/experiments/requirements-gpu.txt")
sh(
    f"uv pip install -q --python {PYTHON} --only-binary :all: torch-scatter==2.1.2 "
    "-f https://data.pyg.org/whl/torch-2.8.0+cu126.html"
)
sh(f"uv pip install -q --python {PYTHON} --no-deps .", cwd=REPO / "graph-walker")
sh(f"{PYTHON} -c 'import torch, graph_walker; print(torch.__version__, torch.cuda.get_device_name(0))'", cwd=WORK)
sh(f"uv pip freeze --python {PYTHON} > {OUTPUT}/e0_gpu_freeze.txt")

In [ ]:
sh(
    f"{PYTHON} check_walker.py --output {OUTPUT}/e0_walker_check_kaggle.json "
    f"2>&1 | tee {OUTPUT}/e0_walker_check_kaggle.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 8 --data-root {WORK}/data "
    "--datasets FB15k237Inductive:v1,WN18RRInductive:v1,NELLInductive:v1 "
    f"--output {OUTPUT}/e0_zero_shot.json 2>&1 | tee {OUTPUT}/e0_zero_shot.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 1 --max-triples 20 --data-root {WORK}/data "
    f"--datasets Hetionet --output {OUTPUT}/e0_hetionet_timing.json 2>&1 | tee {OUTPUT}/e0_hetionet_timing.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
for name in ("e0_zero_shot.json", "e0_hetionet_timing.json"):
    path = OUTPUT / name
    if not path.exists():
        print(name, "is missing; check its log")
        continue
    for run in json.loads(path.read_text())["runs"]:
        print(
            run["dataset"], "| MRR", round(run["measured"]["mrr"], 3), "| paper", run["paper_all_relations"]["mrr"],
            "| full run", run["full_run"], "| s/triple", round(run["seconds_per_triple"], 2),
            "| peak GPU GB", round(run["peak_gpu_memory_gb"], 2),
        )

In [ ]:
if ON_COLAB:
    import shutil

    from google.colab import files

    files.download(shutil.make_archive("/content/e0_results", "zip", OUTPUT))